# corr_hbo_velmed.ipynb\n\n**Purpose:** Spearman correlation between the longitudinal slope of HbO beta\nand the longitudinal slope of reading time (VELmed) for each of the 6 FDR-significant\nfNIRS channels identified by channel_lmm_anatomical.ipynb.\n\n**Inputs:**\n- subjstats_completo.xlsx — fNIRS GLM betas\n- sessao_stats.xlsx — behavioral data\n\n**Outputs:**\n- corr_hbo_velmed.xlsx — Spearman rho, p-value (FDR-corrected) per channel × scope\n\n**Method:**\n- Per subject: OLS slope of beta ~ sessao_num (sessions 1-9)\n- Per subject: OLS slope of VELmed ~ sessao_num (sessions 1-9)\n- Spearman(hbo_slope, VELmed_slope) for 3 scopes: all / acelerado / nao_acelerado\n- FDR Benjamini-Hochberg over 18 tests (6 channels × 3 scopes)\n- N = 13 (SUBJ_030 excluded as outlier)\n\n**Library:** statsmodels 0.14.6, scipy (Python 3.12)\n\n**Author:** Lucas Gemal (lucasgemal@gmail.com) — IDOR / UFRJ

# Correlation: HbO Slope × VELmed Slope
**Project SESI**

For each significant channel: Spearman correlation between  
- **HbO slope** (β of `hbo ~ sessao_num` per subject)  
- **VELmed slope** (β of `VELmed ~ sessao_num` per subject)  

Channels tested: significant after FDR from channel-level LMM (HbO)  
N = 14 subjects per correlation

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
import warnings
warnings.filterwarnings('ignore')

## 1 — Configuration

In [ ]:
# Update this path to match your local data directory
PATH_FNIRS = r'../data/subjstats_individuais.xlsx'
# Update this path to match your local data directory
PATH_BEH   = r'../data/sessao_stats.xlsx'

# Update this path to match your local data directory
BASE_OUT       = r'../results/corr_hbo_velmed'
PATH_OUT_EXCEL = f'{BASE_OUT}.xlsx'
PATH_OUT_FIG   = f'{BASE_OUT}_scatter.png'

# Significant channels after FDR (HbO) — from channel LMM results
# Format: channel: (10-10 name, anatomical region, ROI)
SIG_CHANNELS = {
    'S3-D1': ('AF3-F5',  'L IFG p.Triangularis (22%)',    'FRONTAL'),
    'S5-D4': ('P7-TP7',  'L Middle Temporal Gyrus (41%)', 'TEMPORAL'),
    'S7-D7': ('T7-C5',   'L Middle Temporal Gyrus (55%)', 'TEMPORAL'),
    'S5-D6': ('P7-P5',   'L Middle Temporal Gyrus (47%)', 'TEMPORAL'),
    'S5-D5': ('P7-P9',   'L Inferior Temporal Gyrus (29%)', 'TEMPORAL'),
    'S7-D4': ('CP5-TP7', 'L Middle Temporal Gyrus (72%)', 'TEMPORAL'),
}

PALETTE = {
    'acelerado'    : 'darkorange',
    'nao_acelerado': 'steelblue'
}

print(f'Channels to correlate: {len(SIG_CHANNELS)}')
for ch, (name, region, roi) in SIG_CHANNELS.items():
    print(f'  {ch} ({name}) — {region} [{roi}]')

## 2 — Load fNIRS and compute HbO slope per subject × channel

In [ ]:
df_fnirs = pd.read_excel(PATH_FNIRS)
df_hbo   = df_fnirs[df_fnirs['type'] == 'hbo'].copy()

# Extract session number
def parse_session(exp):
    exp = str(exp).strip().lower()
    if 'calib' in exp: return 0
    m = re.search(r'(\d+)', exp)
    return int(m.group(1)) if m else -1

df_hbo['sessao_num'] = df_hbo['experiment'].apply(parse_session)

# Keep only significant channels, training sessions only (1-9)
df_hbo = df_hbo[
    (df_hbo['channel'].isin(SIG_CHANNELS.keys())) &
    (df_hbo['sessao_num'] >= 1)
].copy()

# Compute OLS slope of hbo ~ sessao_num per subject × channel
hbo_slopes = []
for channel in SIG_CHANNELS:
    for subject in df_hbo['subject'].unique():
        sub = df_hbo[
            (df_hbo['channel'] == channel) &
            (df_hbo['subject'] == subject)
        ]
        if len(sub) < 3: continue
        try:
            model = smf.ols('beta ~ sessao_num', data=sub).fit()
            hbo_slopes.append({
                'subject'   : subject,
                'group'     : sub['group'].iloc[0],
                'channel'   : channel,
                'hbo_slope' : model.params.get('sessao_num', np.nan)
            })
        except Exception:
            pass

df_hbo_slopes = pd.DataFrame(hbo_slopes)
print(f'HbO slopes computed: {len(df_hbo_slopes)} rows')
print(f'Subjects: {df_hbo_slopes["subject"].nunique()}')

## 3 — Load behavioral and compute VELmed slope per subject

In [ ]:
df_beh = pd.read_excel(PATH_BEH)
df_beh['subj_num'] = df_beh['SUBJID'].str.extract(r'(\d+)').astype(int)

# Melt VELmed columns wide → long (sessions 1-9 only)
vel_cols = [c for c in df_beh.columns if c.startswith('VELmed')]
df_vel = df_beh.melt(
    id_vars=['subj_num', 'grupo'],
    value_vars=vel_cols,
    var_name='vel_col', value_name='VELmed'
)
df_vel['sessao_num'] = df_vel['vel_col'].str.extract(r'(\d+)').astype(int)
df_vel = df_vel[df_vel['sessao_num'] >= 1].copy()

# Normalize subject ID to match fNIRS (subj_num → SUBJ_XXX)
df_vel['subject'] = df_vel['subj_num'].apply(lambda x: f'SUBJ_{x:03d}')

# Compute OLS slope of VELmed ~ sessao_num per subject
vel_slopes = []
for subject in df_vel['subject'].unique():
    sub = df_vel[df_vel['subject'] == subject]
    if len(sub) < 3: continue
    try:
        model = smf.ols('VELmed ~ sessao_num', data=sub).fit()
        vel_slopes.append({
            'subject'    : subject,
            'velmed_slope': model.params.get('sessao_num', np.nan)
        })
    except Exception:
        pass

df_vel_slopes = pd.DataFrame(vel_slopes)
print(f'VELmed slopes computed: {len(df_vel_slopes)} subjects')
print(df_vel_slopes[['subject','velmed_slope']].to_string(index=False))

In [ ]:
# Verifica estrutura dos dois DataFrames antes de mergear
print("=== df_hbo_slopes ===")
print(df_hbo_slopes.shape)
print(df_hbo_slopes.columns.tolist())
print(df_hbo_slopes.head(3))

print("\n=== df_vel_slopes ===")
print(df_vel_slopes.shape)
print(df_vel_slopes.columns.tolist())
print(df_vel_slopes.head(3))

# Merge: cada linha de hbo_slopes ganha o velmed_slope do sujeito
df_merged = pd.merge(df_hbo_slopes, df_vel_slopes, on='subject', how='inner')

print(f"df_merged shape: {df_merged.shape}")
print(f"Sujeitos: {df_merged['subject'].nunique()}")
print(f"Canais únicos: {df_merged['channel'].nunique()}")
print(df_merged.head(3))


## 4 — Merge and compute Spearman correlations

In [ ]:
# Cole numa célula nova e execute
print("Variáveis disponíveis no namespace:")
variaveis = [v for v in dir() if not v.startswith('_')]
for v in sorted(variaveis):
    try:
        val = eval(v)
        tipo = type(val).__name__
        if hasattr(val, '__len__'):
            print(f"  {v:<30} {tipo:<15} len={len(val)}")
        else:
            print(f"  {v:<30} {tipo}")
    except:
        pass

In [ ]:
# Exclude outlier
EXCLUDE_SUBJECTS = ['SUBJ_030']
df_merged_clean = df_merged[~df_merged['subject'].isin(EXCLUDE_SUBJECTS)].copy()
print(f'Subjects after exclusion: {df_merged_clean["subject"].nunique()}')
print(f'Excluded: {EXCLUDE_SUBJECTS}\n')

# Spearman correlation per channel — three versions:
# A) all subjects minus outlier
# B) accelerated only
# C) non-accelerated only
corr_results = []

for channel, (name_1010, region, roi) in SIG_CHANNELS.items():
    for scope, group_filter in [
        ('all',           None),
        ('accelerated',   'acelerado'),
        ('non_accelerated', 'nao_acelerado')
    ]:
        sub = df_merged_clean[df_merged_clean['channel'] == channel].copy()
        if group_filter:
            sub = sub[sub['group'] == group_filter]
        sub = sub.dropna()
        if len(sub) < 4: continue

        rho, p = spearmanr(sub['hbo_slope'], sub['velmed_slope'])
        corr_results.append({
            'channel'   : channel,
            'name_1010' : name_1010,
            'region'    : region,
            'roi'       : roi,
            'scope'     : scope,
            'n'         : len(sub),
            'rho'       : round(rho, 3),
            'p_value'   : round(p, 4),
            'direction' : 'expected' if rho > 0 else 'unexpected'
        })

df_corr = pd.DataFrame(corr_results)

# FDR per scope
for scope in df_corr['scope'].unique():
    mask = df_corr['scope'] == scope
    pvals = df_corr.loc[mask, 'p_value'].values
    if len(pvals) < 2: continue
    _, q_vals, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
    df_corr.loc[mask, 'q_value'] = q_vals.round(4)

df_corr['sig_p']   = df_corr['p_value'] < 0.05
df_corr['sig_fdr'] = df_corr['q_value'] < 0.05

# Print by scope
for scope in ['all', 'accelerated', 'non_accelerated']:
    sub = df_corr[df_corr['scope'] == scope]
    print(f'\n=== {scope.upper()} (N={sub["n"].iloc[0] if len(sub) > 0 else "?"}) ===')
    print(sub[['channel','region','rho','p_value','q_value','direction','sig_p']]
          .sort_values('p_value').to_string(index=False))

## 5 — Scatter plots per significant channel

In [ ]:
n_ch  = len(SIG_CHANNELS)
ncols = min(3, n_ch)
nrows = int(np.ceil(n_ch / ncols))

fig, axes = plt.subplots(nrows, ncols,
                         figsize=(5*ncols, 4.5*nrows),
                         squeeze=False)

for idx, (channel, (name_1010, region, roi)) in enumerate(SIG_CHANNELS.items()):
    ax  = axes[idx // ncols][idx % ncols]
    sub = df_merged[df_merged['channel'] == channel].dropna()
    row = df_corr[df_corr['channel'] == channel].iloc[0]

    # Scatter by group
    for grp, color in PALETTE.items():
        g = sub[sub['group'] == grp]
        ax.scatter(g['hbo_slope'], g['velmed_slope'],
                   color=color, s=70, edgecolors='black',
                   linewidths=0.5, zorder=5,
                   label='Accelerated' if grp == 'acelerado' else 'Non-Accelerated')

    # Regression line (all subjects)
    x = sub['hbo_slope'].values
    y = sub['velmed_slope'].values
    if len(x) > 2:
        m, b = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, m*x_line + b,
                color='black', linewidth=1.5,
                linestyle='--', alpha=0.7)

    # Stats annotation
    sig_str = ''
    if row['sig_fdr']:   sig_str = '★ FDR'
    elif row['sig_p']:   sig_str = '† p<0.05'
    else:                sig_str = 'n.s.'

    p_str = f"p={row['p_value']:.3f}" if row['p_value'] >= 0.001 else 'p<0.001'
    ax.annotate(
        f"ρ={row['rho']:.3f}, {p_str}\nq={row['q_value']:.3f} {sig_str}",
        xy=(0.05, 0.95), xycoords='axes fraction',
        fontsize=8, va='top',
        bbox=dict(boxstyle='round,pad=0.3',
                  facecolor='white', edgecolor='grey', alpha=0.8)
    )

    ax.axhline(0, linestyle=':', color='grey', alpha=0.5)
    ax.axvline(0, linestyle=':', color='grey', alpha=0.5)
    ax.set_title(f'{region}\n{channel} / {name_1010}',
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('HbO slope (β/session)', fontsize=9)
    ax.set_ylabel('VELmed slope (chars/ms/session)', fontsize=9)
    ax.grid(linestyle='--', alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=8)

# Hide unused axes
for idx in range(n_ch, nrows*ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle(
    'Spearman Correlation: HbO Slope × VELmed Slope\n'
    'Per subject across training sessions (N=14)',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(PATH_OUT_FIG, dpi=300, bbox_inches='tight')
print(f'-> Figure saved: {PATH_OUT_FIG}')
plt.show()

## 6 — Export

In [ ]:
with pd.ExcelWriter(PATH_OUT_EXCEL, engine='openpyxl') as writer:
    df_corr.to_excel(writer,   sheet_name='correlations',  index=False)
    df_merged.to_excel(writer, sheet_name='slopes_merged', index=False)
    df_vel_slopes.to_excel(writer, sheet_name='velmed_slopes', index=False)

print(f'\n✅ Results saved: {PATH_OUT_EXCEL}')
print(f'   Channels tested : {len(df_corr)}')
print(f'   Sig uncorrected : {df_corr["sig_p"].sum()}')
print(f'   Sig after FDR   : {df_corr["sig_fdr"].sum()}')
print('   Sheets: correlations | slopes_merged | velmed_slopes')